# **Task 2 — DeiT**: does seeing more pixels help?

This week's seminar was about [DeiT](https://proceedings.mlr.press/v139/touvron21a.html): a ViT trained on ImageNet alone, with distillation, and fine-tuned at a higher resolution at the end. The paper reports that going from 224×224 to 384×384 images makes the model more accurate (Table 10). Today you check whether that holds for **our** model and **our** data.

This notebook loads the data and the model and gives you a **fixed evaluation**, so that every group measures its result the same way. Your own work starts after the baseline.

<details>
<summary>Reach out here if you have any problems</summary>
Taras Kutsyk — taras.kutsyk@doctoral.uj.edu.pl
</details>

# The task

**Question.** Does seeing more pixels help? Make the model run at 384×384, fine-tune it within the budget, and **answer the question for this model and this data, with evidence**. Then explain *why* you got the result you got, and what they teach you.

*Hint:* be wary of what you compare with, and of how your claim relates to the paper's.

**Fixed:**
- the starting model and the dataset;
- the `evaluate()` function below;
- the fine-tuning budget: **at most 100,000 training images per fine-tuned model** (about 11 passes over the training set);
- write the resizing of the position embeddings yourselves: no ready-made resizing (e.g. timm's `img_size=` argument or `resample_abs_pos_embed`).

Everything else is your choice.

**Bring to the lab:** your code + a short deck with your answer, the evidence for it, and your explanation. If your group defends this week, **every member should be able to explain any piece of the code**.

# Setup

**Step 0.** Runtime → Change runtime type → **T4 GPU**. Everything we need (PyTorch, torchvision, timm) is already installed on Colab.

**Step 1. Data.** [Imagewoof](https://github.com/fastai/imagenette): 10 dog breeds from ImageNet, 9,025 training and 3,929 validation photos, **at full size** (about 1.3 GB, under a minute to download).

In [ ]:
import os, tarfile, time, urllib.request
import torch, timm
from torchvision import datasets

device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "Switch to a GPU runtime: Runtime -> Change runtime type -> T4 GPU"

URL = "https://s3.amazonaws.com/fast-ai-imageclas/imagewoof2.tgz"
# we use the full-size image sets on purpose, not the smaller "-320" versions
# can you tell why? (in the deck)
if not os.path.exists("imagewoof2/val"):
    urllib.request.urlretrieve(URL, "imagewoof2.tgz")
    with tarfile.open("imagewoof2.tgz") as t:
        t.extractall(filter="data")
for split in ["train", "val"]:
    print(split, sum(len(f) for _, _, f in os.walk(f"imagewoof2/{split}")), "photos")

train 9025 photos
val 3929 photos


**Step 2. Model.** A distilled DeiT-S from the paper (12 blocks, 384 dims, 22M parameters, the DeiT-S⚗ of Table 5), trained on ImageNet at 224×224, together with the exact preprocessing it was trained with.

The model predicts one of 1000 ImageNet classes, so we map each Imagewoof folder (e.g. `n02086240` = Shih-Tzu) to its **ImageNet index**: the position of that breed among the model's 1000 outputs. You will need `to_imagenet` for your fine-tuning too.

<details>
<summary>Running the cell below can ask for your HF_TOKEN secret</summary>
Just press "Grant Access", it's automatically generated by Colab
</details>

In [ ]:
model = timm.create_model("deit_small_distilled_patch16_224", pretrained=True).to(device).eval()
cfg = timm.data.resolve_data_config({}, model=model)  # the preprocessing the model was trained with

# ImageFolder labels the 10 folders 0..9, but the model outputs 1000 ImageNet classes:
# to_imagenet[label] translates a folder label into the ImageNet class index
wnids = list(timm.data.ImageNetInfo().label_names())
breeds = datasets.ImageFolder("imagewoof2/val").classes
to_imagenet = torch.tensor([wnids.index(w) for w in breeds], device=device)
print(cfg)
print("folder -> ImageNet index:", dict(zip(breeds, to_imagenet.tolist())))

model.safetensors: reconstructing file:   0%|          |  0.00B / 89.8MB            

model.safetensors: downloading bytes:           |  0.00B            

{'input_size': (3, 224, 224), 'interpolation': 'bicubic', 'mean': (0.485, 0.456, 0.406), 'std': (0.229, 0.224, 0.225), 'crop_pct': 0.9, 'crop_mode': 'center'}
folder -> ImageNet index: {'n02086240': 155, 'n02087394': 159, 'n02088364': 162, 'n02089973': 167, 'n02093754': 182, 'n02096294': 193, 'n02099601': 207, 'n02105641': 229, 'n02111889': 258, 'n02115641': 273}


**Step 3. The fixed evaluation** (DON'T modify it). `evaluate(model, resolution)` resizes the validation photos for the given resolution exactly as the model's own preprocessing does, and returns the **top-1 accuracy over the 10 breeds**: the model's prediction is the highest-scoring of its 10 dog outputs.

In [ ]:
@torch.no_grad()
def evaluate(model, resolution):
    """The fixed evaluation for task 2. Do not change it; report exactly what it prints.

    model      - any model that takes (batch, 3, resolution, resolution) images and returns 1000 logits
    resolution - the input size: 224 for the original model, e.g. 384 for yours
    returns: accuracy - top-1 over the 10 breeds, on the 3,929 Imagewoof validation photos
    """
    model.eval()
    tf = timm.data.create_transform(**dict(cfg, input_size=(3, resolution, resolution)))
    loader = torch.utils.data.DataLoader(datasets.ImageFolder("imagewoof2/val", transform=tf),
                                         batch_size=64, num_workers=2)
    correct = n = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(x)
        correct += (logits[:, to_imagenet].argmax(-1) == y).sum().item()  # best of the 10 breeds
        n += len(y)
    return correct / n

**Step 4. Baseline.** Takes about half a minute. You should get **top-1 ≈ 97.20%** at 224. If not, something in the setup went wrong.

In [ ]:
print(f"baseline DeiT-S distilled @224: top-1 {evaluate(model, 224):.2%}")

baseline DeiT-S distilled @224: top-1 97.20%


# Before you start

Think about **what exactly breaks in the model when you pass it higher-resolution images?** What's the simplest, most effective way to fix that?

Training photos are in `imagewoof2/train` (same folder structure as `val`).

# Your solution

Make the model run at 384, fine-tune within the budget, and collect the evidence for your answer. Good luck!

In [ ]:
# your_model = ...
# print(f"ours @384: top-1 {evaluate(your_model, 384):.2%}")

In [ ]:
# ── CELL 1 ─ Imports & Constants ────────────────────────────
import math, copy, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import timm
from pathlib import Path

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RES        = 384            # target resolution
OLD_GRID   = 14             # 224 // 16  (original patch grid)
NEW_GRID   = RES // 16      # 384 // 16 = 24 (new patch grid)
N_TRAIN    = 9_025          # imagewoof train set size
MAX_IMGS   = 100_000        # image budget
MAX_EPOCHS = MAX_IMGS // N_TRAIN   # 11
BATCH      = 32
print(f"Device: {DEVICE} | target res: {RES} | max epochs: {MAX_EPOCHS}")

Device: cuda | target res: 384 | max epochs: 11


In [ ]:
# ── CELL 2 ─ Class-index mapping ────────────────────────────
# to_imagenet is a Tensor of shape (10,) defined in the starter.
# to_imagenet[i] = ImageNet class index for imagewoof label i.
# We build the inverse: imagenet_idx → woof_label (0–9).

if isinstance(to_imagenet, torch.Tensor):
    imagenet_to_woof = {int(to_imagenet[i]): i for i in range(len(to_imagenet))}
else:
    imagenet_to_woof = {v: k for k, v in to_imagenet.items()}

print("imagenet_to_woof ready:", len(imagenet_to_woof), "entries")
print("Example:", next(iter(imagenet_to_woof.items())))

imagenet_to_woof ready: 10 entries
Przykład: (155, 0)


In [ ]:
# ── CELL 3 ─ Data Loaders @ 384 ─────────────────────────────
IMAGEWOOF = Path("imagewoof2")   # adjust if downloaded elsewhere

transform_train_384 = T.Compose([
    T.RandomResizedCrop(RES, scale=(0.08, 1.0),
                        interpolation=T.InterpolationMode.BICUBIC),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

transform_val_384 = T.Compose([
    # Resize shorter side to 439, then centre-crop to 384
    T.Resize(int(RES * 1.143), interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(RES),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_ds = ImageFolder(IMAGEWOOF / "train", transform=transform_train_384)
val_ds   = ImageFolder(IMAGEWOOF / "val",   transform=transform_val_384)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=64,    shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"Train: {len(train_ds):,} | Val: {len(val_ds):,}")

Train: 9,025 | Val: 3,929


In [ ]:
# ── CELL 4 ─ Positional-Embedding Interpolation (fixed) ─────

def interpolate_pos_embed(model, new_res: int = RES) -> None:
    """Resize PE 14²→24² AND update patch_embed metadata."""
    new_grid = new_res // 16        # 24

    # ── 1. Interpolate pos_embed ─────────────────────────────
    pe       = model.pos_embed.data           # (1, 198, D)
    cls_pe   = pe[:, :1, :]
    dist_pe  = pe[:, 1:2, :]
    patch_pe = pe[:, 2:, :]                   # (1, 196, D)

    D     = patch_pe.shape[-1]
    old_g = int(patch_pe.shape[1] ** 0.5)    # 14

    patch_pe = (patch_pe
                .reshape(1, old_g, old_g, D)
                .permute(0, 3, 1, 2)
                .float())
    patch_pe = F.interpolate(
        patch_pe, size=(new_grid, new_grid),
        mode="bicubic", align_corners=False
    )
    patch_pe = (patch_pe
                .permute(0, 2, 3, 1)
                .reshape(1, new_grid ** 2, D))

    model.pos_embed = nn.Parameter(
        torch.cat([cls_pe, dist_pe, patch_pe], dim=1)
    )

    # ── 2. Update patch_embed metadata ──────────────────────
    model.patch_embed.img_size    = (new_res, new_res)
    model.patch_embed.grid_size   = (new_grid, new_grid)
    model.patch_embed.num_patches = new_grid * new_grid

    print(f"  PE:  {old_g}²={old_g**2} → {new_grid}²={new_grid**2} patches  "
          f"(D={D}, img_size={new_res}×{new_res})")

In [ ]:
# ── CELL 5 ─ Loss Functions ──────────────────────────────────

class DualHeadCELoss(nn.Module):
    """
    L = ½ CE(cls_head, y) + ½ CE(dist_head, y)

    Used for Solutions B1 and B2 (no teacher).
    Both heads learn directly from ground-truth labels.
    Label smoothing ε=0.1 prevents over-confident predictions
    on the small 9 k training set.
    """
    def __init__(self, label_smoothing: float = 0.1):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    def forward(self, cls_logits, dist_logits, labels):
        return 0.5 * self.ce(cls_logits, labels) \
             + 0.5 * self.ce(dist_logits, labels)


class HardDistillationLoss(nn.Module):
    """
    L = ½ CE(cls_head,  y_true)    ← smoothed CE vs ground truth
      + ½ CE(dist_head, y_teacher) ← hard CE vs teacher's top-1

    Used for Solution D.

    y_teacher = argmax(teacher(x_224)) mapped to imagewoof space
    via `imagenet_to_woof`.  Falls back to y_true if the teacher
    predicts a class outside imagewoof (e.g. "Chihuahua").

    Why two heads?
      The CLS  token aggregates global context → generalises well.
      The DIST token was designed to mimic the teacher's output
      through attention; it learns complementary features.
      At inference both heads are averaged, boosting accuracy.
    """
    def __init__(self, label_smoothing: float = 0.1):
        super().__init__()
        self.ce_smooth = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
        self.ce_hard   = nn.CrossEntropyLoss()   # no smoothing vs teacher

    def forward(self, cls_logits, dist_logits, teacher_1k_logits, labels):
        # Map teacher's ImageNet top-1 prediction → imagewoof label (0–9)
        teacher_imagenet = teacher_1k_logits.argmax(dim=-1).cpu().tolist()
        teacher_woof = torch.tensor(
            [imagenet_to_woof.get(idx, int(labels[i]))   # fallback = GT
             for i, idx in enumerate(teacher_imagenet)],
            dtype=torch.long, device=labels.device
        )
        loss_cls  = self.ce_smooth(cls_logits,  labels)
        loss_dist = self.ce_hard(dist_logits, teacher_woof)
        return 0.5 * loss_cls + 0.5 * loss_dist

In [ ]:
# ── CELL 6 ─ Training Utilities ─────────────────────────────

def count_params(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Params: {trainable:,} trainable / {total:,} total "
          f"({100 * trainable / total:.2f}%)")
    return trainable


@torch.no_grad()
def val_accuracy(model, loader=val_loader, device=DEVICE):
    """Evaluate model; distilled DeiT averages both heads in eval mode."""
    model.eval()
    correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)           # timm returns averaged logits in eval
        if isinstance(logits, (tuple, list)):
            logits = (logits[0] + logits[1]) * 0.5
        correct += (logits.argmax(dim=-1) == y).sum().item()
        total   += y.size(0)
    return 100.0 * correct / total


def train_one_epoch(model, criterion, optimizer, scaler,
                    teacher=None, device=DEVICE):
    model.train()
    total_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda"):
            out = model(x)
            # DeiT-distilled returns tuple (cls, dist) during training
            cls_logits, dist_logits = out if isinstance(out, tuple) else (out, out)

            if teacher is not None:
                with torch.no_grad():
                    # Resize input to 224 for the teacher (cheap bilinear)
                    x_224 = F.interpolate(x, size=224, mode="bilinear",
                                          align_corners=False)
                    teacher_logits = teacher(x_224)   # (B, 1000)
                loss = criterion(cls_logits, dist_logits, teacher_logits, y)
            else:
                loss = criterion(cls_logits, dist_logits, y)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()

    return total_loss / len(train_loader)


def run_training(model, label, criterion, optimizer, scheduler,
                 n_epochs=MAX_EPOCHS, teacher=None):
    scaler     = torch.amp.GradScaler("cuda")
    best_acc   = 0.0
    best_state = None
    t0         = time.time()

    for ep in range(1, n_epochs + 1):
        avg_loss = train_one_epoch(model, criterion, optimizer, scaler,
                                   teacher=teacher)
        acc = val_accuracy(model)
        scheduler.step()

        if acc > best_acc:
            best_acc   = acc
            best_state = copy.deepcopy(model.state_dict())

        elapsed = (time.time() - t0) / 60
        print(f"[{label}] ep {ep:2d}/{n_epochs}  "
              f"loss={avg_loss:.4f}  val={acc:.2f}%  "
              f"best={best_acc:.2f}%  ({elapsed:.1f} min)")

    model.load_state_dict(best_state)
    print(f"→ [{label}] Best val: {best_acc:.2f}%\n")
    return best_acc

In [ ]:
# ════════════════════════════════════════════════════════════
#  SOLUTION B1  —  Linear Probe @ 384
#  Trainable: head + head_dist  (≈ 2×384×10 = 7 680 params)
#  Backbone + pos_embed: fully frozen
# ════════════════════════════════════════════════════════════

# ── CELL 7 ──────────────────────────────────────────────────
print("=" * 56)
print("SOLUTION B1  —  Linear Probe (head + head_dist only)")
print("=" * 56)

model_b1 = timm.create_model(
    "deit_small_distilled_patch16_224",
    pretrained=True, num_classes=10
).to(DEVICE)

# Step 1 — interpolate PE from 14×14 to 24×24
interpolate_pos_embed(model_b1)

# Step 2 — freeze ALL parameters
for p in model_b1.parameters():
    p.requires_grad_(False)

# Step 3 — unfreeze only the two linear classifier heads
for p in model_b1.head.parameters():       # CLS  head  (384 → 10)
    p.requires_grad_(True)
for p in model_b1.head_dist.parameters():  # DIST head  (384 → 10)
    p.requires_grad_(True)

count_params(model_b1)

# High LR is safe: only 2 small linear layers, no backbone risk
opt_b1 = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model_b1.parameters()),
    lr=1e-3, weight_decay=0.05
)
sch_b1  = torch.optim.lr_scheduler.CosineAnnealingLR(opt_b1, MAX_EPOCHS, eta_min=1e-5)
crit_b1 = DualHeadCELoss(label_smoothing=0.1)

acc_b1 = run_training(model_b1, "B1", crit_b1, opt_b1, sch_b1)

SOLUTION B1  —  Linear Probe (head + head_dist only)
  PE:  14²=196 → 24²=576 patches  (D=384, img_size=384×384)
  Params: 7,700 trainable / 21,820,052 total (0.04%)
[B1] ep  1/11  loss=0.7964  val=96.54%  best=96.54%  (3.7 min)
[B1] ep  2/11  loss=0.7237  val=96.79%  best=96.79%  (7.6 min)
[B1] ep  3/11  loss=0.7202  val=96.72%  best=96.79%  (11.4 min)
[B1] ep  4/11  loss=0.7084  val=96.69%  best=96.79%  (15.5 min)
[B1] ep  5/11  loss=0.7060  val=96.77%  best=96.79%  (19.4 min)
[B1] ep  6/11  loss=0.6984  val=96.72%  best=96.79%  (23.3 min)
[B1] ep  7/11  loss=0.6938  val=96.97%  best=96.97%  (27.2 min)
[B1] ep  8/11  loss=0.6949  val=97.12%  best=97.12%  (31.2 min)
[B1] ep  9/11  loss=0.6874  val=97.00%  best=97.12%  (35.1 min)
[B1] ep 10/11  loss=0.6967  val=97.00%  best=97.12%  (39.1 min)
[B1] ep 11/11  loss=0.6869  val=97.05%  best=97.12%  (43.1 min)
→ [B1] Best val: 97.12%



In [ ]:
# ════════════════════════════════════════════════════════════
#  SOLUTION B2  —  Linear Probe + pos_embed @ 384
#  Trainable: head + head_dist + pos_embed
#
#  WHY unfreeze pos_embed here?
#  Bicubic interpolation gives geometrically consistent values,
#  but the frozen attention layers were never trained with 24×24
#  positions — the spatial "grammar" is mismatched.
#  Letting pos_embed fine-tune with a small LR corrects that
#  mismatch without touching the backbone weights, which would
#  risk overfitting on our tiny 9 k dataset.
# ════════════════════════════════════════════════════════════

# ── CELL 8 ──────────────────────────────────────────────────
print("=" * 56)
print("SOLUTION B2  —  Linear Probe + pos_embed @ 384")
print("=" * 56)

model_b2 = timm.create_model(
    "deit_small_distilled_patch16_224",
    pretrained=True, num_classes=10
).to(DEVICE)

interpolate_pos_embed(model_b2)

# Freeze everything
for p in model_b2.parameters():
    p.requires_grad_(False)

# Unfreeze: heads + pos_embed
for p in model_b2.head.parameters():
    p.requires_grad_(True)
for p in model_b2.head_dist.parameters():
    p.requires_grad_(True)
model_b2.pos_embed.requires_grad_(True)     # ← the key difference vs B1

count_params(model_b2)

# Separate LR groups: lower rate for pos_embed to stay conservative
opt_b2 = torch.optim.AdamW([
    {"params": model_b2.head.parameters(),       "lr": 1e-3},
    {"params": model_b2.head_dist.parameters(),  "lr": 1e-3},
    {"params": [model_b2.pos_embed],              "lr": 1e-4},  # ← 10× smaller
], weight_decay=0.05)
sch_b2  = torch.optim.lr_scheduler.CosineAnnealingLR(opt_b2, MAX_EPOCHS, eta_min=1e-6)
crit_b2 = DualHeadCELoss(label_smoothing=0.1)

acc_b2 = run_training(model_b2, "B2", crit_b2, opt_b2, sch_b2)


SOLUTION B2  —  Linear Probe + pos_embed @ 384
  PE:  14²=196 → 24²=576 patches  (D=384, img_size=384×384)
  Params: 229,652 trainable / 21,820,052 total (1.05%)
[B2] ep  1/11  loss=0.7865  val=96.72%  best=96.72%  (4.0 min)
[B2] ep  2/11  loss=0.7111  val=96.87%  best=96.87%  (8.0 min)
[B2] ep  3/11  loss=0.6966  val=97.07%  best=97.07%  (12.1 min)
[B2] ep  4/11  loss=0.6870  val=96.89%  best=97.07%  (16.0 min)
[B2] ep  5/11  loss=0.6866  val=96.77%  best=97.07%  (20.1 min)
[B2] ep  6/11  loss=0.6812  val=96.82%  best=97.07%  (24.1 min)
[B2] ep  7/11  loss=0.6798  val=96.74%  best=97.07%  (28.1 min)
[B2] ep  8/11  loss=0.6709  val=97.02%  best=97.07%  (32.2 min)
[B2] ep  9/11  loss=0.6692  val=97.02%  best=97.07%  (36.4 min)
[B2] ep 10/11  loss=0.6726  val=97.00%  best=97.07%  (40.5 min)
[B2] ep 11/11  loss=0.6694  val=97.02%  best=97.07%  (44.5 min)
→ [B2] Best val: 97.07%



In [ ]:
# ════════════════════════════════════════════════════════════
#  SOLUTION D  —  Layer Freezing @ 384
#
#  Architecture:
#    patch_embed + blocks 0–7  → FROZEN   (8 / 12 blocks)
#    pos_embed + blocks 8–11   → trainable
#    norm + head + head_dist   → trainable
#
#  Loss: Hard Knowledge Distillation
#    Teacher = DeiT-S @224 (pretrained, 1 000 classes, frozen)
#    Student = DeiT-S @384 (blocks 8–11 + PE + heads trainable)
#
#  WHY freeze early blocks?
#    Blocks 0–7 learn low-level features (edges, textures, local
#    patches) that are largely resolution-agnostic and already
#    well-trained on ImageNet.  Keeping them frozen:
#      (a) cuts trainable params → far less overfitting risk
#          on our 9 025-image dataset;
#      (b) preserves the strong ImageNet representations.
#    Blocks 8–11 learn high-level, global semantics that benefit
#    from 384×384's extra context — they need to adapt.
# ════════════════════════════════════════════════════════════

# ── CELL 9 ──────────────────────────────────────────────────
print("=" * 56)
print("SOLUTION D  —  Layer Freezing (blocks 8–11 + PE + heads)")
print("=" * 56)

# ── Teacher (frozen DeiT-S, 1 000 ImageNet classes) ─────────
teacher = timm.create_model(
    "deit_small_distilled_patch16_224",
    pretrained=True, num_classes=1000
).to(DEVICE).eval()
for p in teacher.parameters():
    p.requires_grad_(False)
print("Teacher loaded: DeiT-S @224, 1 000 classes, fully frozen.\n")

# ── Student model ────────────────────────────────────────────
model_d = timm.create_model(
    "deit_small_distilled_patch16_224",
    pretrained=True, num_classes=10
).to(DEVICE)

interpolate_pos_embed(model_d)

# ── Selective requires_grad ──────────────────────────────────
FREEZE_BLOCKS = 8   # freeze blocks 0 … 7  (first 8 of 12)

# Freeze: patch embedding conv, CLS token, DIST token
for p in model_d.patch_embed.parameters():
    p.requires_grad_(False)
model_d.cls_token.requires_grad_(False)
model_d.dist_token.requires_grad_(False)

# Freeze/unfreeze transformer blocks
print("Block freeze status:")
for i, block in enumerate(model_d.blocks):
    frozen = (i < FREEZE_BLOCKS)
    for p in block.parameters():
        p.requires_grad_(not frozen)
    print(f"  block {i:2d}: {'FROZEN   ' if frozen else 'trainable'}")

# Unfreeze pos_embed, final LayerNorm, both heads
model_d.pos_embed.requires_grad_(True)
for p in model_d.norm.parameters():
    p.requires_grad_(True)
for p in model_d.head.parameters():
    p.requires_grad_(True)
for p in model_d.head_dist.parameters():
    p.requires_grad_(True)

print()
count_params(model_d)

# ── Differential learning rates ──────────────────────────────
# pos_embed + backbone tail → moderate (pre-trained weights, gentle update)
# heads                     → higher   (newly initialised 10-class layers)
opt_d = torch.optim.AdamW([
    {"params": [model_d.pos_embed],
                                              "lr": 5e-5},
    {"params": model_d.norm.parameters(),     "lr": 5e-5},
    {"params": [p for blk in model_d.blocks[FREEZE_BLOCKS:]
                  for p in blk.parameters()], "lr": 5e-5},
    {"params": model_d.head.parameters(),     "lr": 5e-4},
    {"params": model_d.head_dist.parameters(),"lr": 5e-4},
], weight_decay=0.05)

sch_d  = torch.optim.lr_scheduler.CosineAnnealingLR(opt_d, MAX_EPOCHS, eta_min=1e-6)
crit_d = HardDistillationLoss(label_smoothing=0.1)

acc_d = run_training(model_d, "D", crit_d, opt_d, sch_d, teacher=teacher)


SOLUTION D  —  Layer Freezing (blocks 8–11 + PE + heads)
Teacher loaded: DeiT-S @224, 1 000 classes, fully frozen.

  PE:  14²=196 → 24²=576 patches  (D=384, img_size=384×384)
Block freeze status:
  block  0: FROZEN   
  block  1: FROZEN   
  block  2: FROZEN   
  block  3: FROZEN   
  block  4: FROZEN   
  block  5: FROZEN   
  block  6: FROZEN   
  block  7: FROZEN   
  block  8: trainable
  block  9: trainable
  block 10: trainable
  block 11: trainable

  Params: 7,328,276 trainable / 21,820,052 total (33.59%)
[D] ep  1/11  loss=0.5832  val=96.26%  best=96.26%  (4.2 min)
[D] ep  2/11  loss=0.4648  val=96.03%  best=96.26%  (8.3 min)
[D] ep  3/11  loss=0.4304  val=96.74%  best=96.74%  (12.5 min)
[D] ep  4/11  loss=0.4286  val=96.79%  best=96.79%  (16.6 min)
[D] ep  5/11  loss=0.4222  val=96.61%  best=96.79%  (20.9 min)
[D] ep  6/11  loss=0.4132  val=97.02%  best=97.02%  (25.0 min)
[D] ep  7/11  loss=0.3997  val=96.95%  best=97.02%  (29.0 min)
[D] ep  8/11  loss=0.3966  val=96.95%  be

In [ ]:
# ════════════════════════════════════════════════════════════
#  RESULTS SUMMARY
# ════════════════════════════════════════════════════════════

# ── CELL 10 ─────────────────────────────────────────────────
print("=" * 56)
print("RESULTS SUMMARY")
print("=" * 56)
BASELINE = 97.20   # starter notebook: DeiT-S @224, no FT

rows = [
    ("Baseline DeiT-S @224 (no FT)",          BASELINE),
    ("B1  Linear probe @384 (head+head_dist)", acc_b1),
    ("B2  Linear probe @384 (+pos_embed)",     acc_b2),
    ("D   Layer-frozen FT @384 (blks 8–11)",   acc_d),
]
for name, acc in rows:
    delta = acc - BASELINE
    sign  = "+" if delta >= 0 else ""
    print(f"  {name:<42s}  {acc:.2f}%  ({sign}{delta:.2f}%)")

print(f"\n  Budget used: {MAX_EPOCHS * N_TRAIN:,} / {MAX_IMGS:,} images")
print(f"  {'Does seeing more pixels help?':}")
best = max(acc_b1, acc_b2, acc_d)
print(f"  Best @384: {best:.2f}%  vs baseline @224: {BASELINE:.2f}%  "
      f"→ {'YES ✓' if best > BASELINE else 'NO ✗'}")

RESULTS SUMMARY
  Baseline DeiT-S @224 (no FT)                97.20%  (+0.00%)
  B1  Linear probe @384 (head+head_dist)      97.12%  (-0.08%)
  B2  Linear probe @384 (+pos_embed)          97.07%  (-0.13%)
  D   Layer-frozen FT @384 (blks 8–11)        97.15%  (-0.05%)

  Budget used: 99,275 / 100,000 images
  Does seeing more pixels help?
  Best @384: 97.15%  vs baseline @224: 97.20%  → NO ✗
